# **Utility Functions**

In [ ]:
pip install split-folders

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
import torch
from pathlib import Path
import os
import splitfolders
import seaborn as sns
import tensorflow as tf
import cv2
import itertools
import os.path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Conv2D, Flatten, Dropout, Dense, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img, img_to_array 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Capstone"

In [ ]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

device = get_default_device()
device

# **Binary Classification**

In [ ]:
!unzip -q "{PROJECT_ROOT}/binary_data.zip" -d "{PROJECT_ROOT}/binary_data"

In [ ]:
def split():
    data_dir = os.path.join(f'{PROJECT_ROOT}/binary_data/data')
    print("No of images in each class of data directory")
    for dir, subdir, files in os.walk(data_dir):
        print(dir,':', str(len(files)))
        
    splitfolders.ratio(f"{PROJECT_ROOT}/binary_data/data", output = f"{PROJECT_ROOT}/binary_data_folders", seed = 6, 
                       ratio = (0.8, 0.2), group_prefix = None, move = False)

In [ ]:
if os.path.exists(f"{PROJECT_ROOT}/binary_data_folders"):
    print("Files already present in splitted format")
else:
    split()
    print("Files are splitted in the ratio 0.7, 0.1, 0.2")

In [ ]:
binary_train = f"{PROJECT_ROOT}/binary_data_folders/train"
binary_val = f"{PROJECT_ROOT}/binary_data_folders/val"

In [ ]:
width, height = 310, 310

In [ ]:
trainDataGen = ImageDataGenerator(rescale = 1./255, rotation_range = 5, zoom_range = 0.1)
validationDataGen = ImageDataGenerator(rescale = 1./255.)
train_generator = trainDataGen.flow_from_directory(binary_train, target_size = (width, height), batch_size = 2, class_mode = 'binary', shuffle = True)
val_generator = validationDataGen.flow_from_directory(binary_val, target_size = (width, height), batch_size = 1, class_mode = 'binary', shuffle = True)

In [ ]:
binary_model = Sequential()
binary_model.add(Conv2D(32, (3, 3), input_shape = (width, height, 3), activation = 'relu'))
binary_model.add(MaxPooling2D(pool_size = (2, 2)))
binary_model.add(Dropout(0.2))

binary_model.add(Conv2D(64, (3, 3), activation = 'relu'))
binary_model.add(MaxPooling2D(pool_size = (2, 2)))
binary_model.add(Dropout(0.2))

binary_model.add(Conv2D(128, (3, 3), activation = 'relu'))
binary_model.add(MaxPooling2D(pool_size = (2, 2)))
binary_model.add(Dropout(0.2))

binary_model.add(Conv2D(128, (3, 3), activation = 'relu'))
binary_model.add(MaxPooling2D(pool_size = (2, 2)))
binary_model.add(Dropout(0.2))

binary_model.add(Flatten())
binary_model.add(Dropout(0.3))
binary_model.add(Dense(units = 512, activation = 'relu'))
binary_model.add(Dropout(0.3))
binary_model.add(Dense(1, activation = 'sigmoid'))

In [ ]:
binary_model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['acc']) 

In [ ]:
history = binary_model.fit(train_generator, epochs = 20, validation_data = val_generator)

In [ ]:
binary_model.save(f"{PROJECT_ROOT}/binary_model.keras")

# **Multiclass Classification**

In [ ]:
!unzip -q "{PROJECT_ROOT}/data_categorised_rgb" -d "{PROJECT_ROOT}/multiclass_data"

In [ ]:
def split():
    data_dir = os.path.join(f'{PROJECT_ROOT}/multiclass_data/data_categorised_rgb')
    print("No of images in each class of data directory")
    for dir, subdir, files in os.walk(data_dir):
        print(dir,':', str(len(files)))
        
    splitfolders.ratio(f"{PROJECT_ROOT}/multiclass_data/data_categorised_rgb", 
                       output = f"{PROJECT_ROOT}/multiclass_data_folders", seed = 6, 
                       ratio = (0.8, 0.2), group_prefix = None, move = False)

In [ ]:
if os.path.exists(f"{PROJECT_ROOT}/Cyclone-Data/multiclass_data_folders"):
    print("Files already present in splitted format")
else:
    split()
    print("Files are splitted in the ratio 0.8, 0.2")

In [ ]:
multiclass_train = PROJECT_ROOT + '/Cyclone-Data/multiclass_data_folders/train'
multiclass_val = PROJECT_ROOT + '/Cyclone-Data/multiclass_data_folders/val'

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1.0/255., rotation_range = 10)
train_generator = train_datagen.flow_from_directory(multiclass_train, batch_size = 2, class_mode = 'categorical', target_size = (310, 310), shuffle = True)
val_datagen = ImageDataGenerator(rescale = 1.0/255.) 
val_generator = val_datagen.flow_from_directory(multiclass_val, batch_size = 1, class_mode = 'categorical', target_size = (310, 310), shuffle = False)

In [ ]:
multiclass_model = Sequential([
tf.keras.layers.Conv2D(32, (3, 3), input_shape = (310, 310, 3), activation = 'relu', name = 'Conv1'),
tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
tf.keras.layers.Dropout(0.2),

tf.keras.layers.Conv2D(64, (3, 3), activation = 'relu', name = 'Conv2'),
tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
tf.keras.layers.Dropout(0.2),

tf.keras.layers.Conv2D(128, (3, 3), activation = 'relu', name = 'Conv3'),
tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
tf.keras.layers.Dropout(0.2),

tf.keras.layers.Conv2D(128, (3, 3), activation = 'relu', name = 'Conv4'),
tf.keras.layers.MaxPooling2D(pool_size = (2, 2)),
tf.keras.layers.Dropout(0.2),

tf.keras.layers.Flatten(),

tf.keras.layers.Dropout(0.4),

tf.keras.layers.Dense(units = 512, activation = 'relu', name = 'FC1'),

tf.keras.layers.Dropout(0.3),

tf.keras.layers.Dense(5, activation = 'softmax', name = 'FC2')
])

In [ ]:
multiclass_model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'categorical_crossentropy', metrics = ['acc'])

In [ ]:
checkpointer = ModelCheckpoint(filepath = f'{PROJECT_ROOT}/multiclass.keras', monitor = 'val_accuracy', verbose = 1, save_best_only = True, mode = 'max')
reduce_lr = ReduceLROnPlateau(monitor = 'val_acc', factor = 0.05, patience = 7, min_lr = 0.00001)

In [ ]:
history = multiclass_model.fit(train_generator, epochs = 30, verbose = 1, validation_data = val_generator, callbacks = [reduce_lr, checkpointer])

In [ ]:
multiclass_model.save(f'{PROJECT_ROOT}/multiclass_model.keras')

# **Detection**

In [ ]:
from PIL import Image

In [ ]:
labels = open(f'{PROJECT_ROOT}/yolo_custom_model_training/custom_data/classes.names').read().strip().split('\n')
weights_path = f'{PROJECT_ROOT}/yolov3_custom_final.weights'
configuration_path = f'{PROJECT_ROOT}/yolov3_custom.cfg'
probability_minimum = 0.3
threshold = 0.3
network = cv2.dnn.readNetFromDarknet(configuration_path, weights_path)
layers_names_output = ['yolo_82', 'yolo_94', 'yolo_106']

In [ ]:
def yolov3(img_path):
    image_input = cv2.imread(img_path)
    image_input_shape = image_input.shape
    plt.rcParams['figure.figsize'] = (10.0, 10.0)
    blob = cv2.dnn.blobFromImage(image_input, 1 / 255.0, (416, 416), swapRB = True, crop = False)
    blob_to_show = blob[0, :, :, :].transpose(1, 2, 0)
    network.setInput(blob)
    output_from_network = network.forward(layers_names_output)
    np.random.seed(42)
    colours = np.random.randint(0, 255, size=(len(labels), 3), dtype='uint8')
    bounding_boxes = []
    confidences = []
    class_numbers = []
    h, w = image_input_shape[:2]
    for result in output_from_network:
        for detection in result:
            scores = detection[5:]
            class_current = np.argmax(scores)
            confidence_current = scores[class_current]
            if confidence_current > probability_minimum:
                box_current = detection[0:4] * np.array([w, h, w, h])
                x_center, y_center, box_width, box_height = box_current.astype('int')
                x_min = int(x_center - (box_width / 2))
                y_min = int(y_center - (box_height / 2))
                bounding_boxes.append([x_min, y_min, int(box_width), int(box_height)])
                confidences.append(float(confidence_current))
                class_numbers.append(class_current)
    results = cv2.dnn.NMSBoxes(bounding_boxes, confidences, probability_minimum, threshold)
    global r
    r = results
    if len(results) > 0 and results is not None:
        for i in results.flatten():
            x_min, y_min = bounding_boxes[i][0], bounding_boxes[i][1]
            box_width, box_height = bounding_boxes[i][2], bounding_boxes[i][3]
            colour_box_current = [int(j) for j in colours[class_numbers[i]]]
            cv2.rectangle(image_input, (x_min, y_min), (x_min + box_width, y_min + box_height), colour_box_current, 5)
            text_box_current = '{}: {:.4f}'.format(labels[int(class_numbers[i])], confidences[i])
            cv2.putText(image_input, text_box_current, (x_min, y_min - 7), cv2.FONT_HERSHEY_PLAIN, 0.9, colour_box_current, 1)

        crop_img = image_input[y_min:y_min+box_height, x_min:x_min + box_width]
        cv2.imwrite(f'{PROJECT_ROOT}/result.jpg', crop_img)
    else:
      print('Cyclone not detected via yolov3')


    plt.rcParams['figure.figsize'] = (15.0, 15.0)
    plt.imshow(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
    plt.show()

# **Regression**

In [ ]:
!unzip -q "{PROJECT_ROOT}/regression_data.zip" -d "{PROJECT_ROOT}/regression_data"

In [ ]:
image_dir = Path(f"{PROJECT_ROOT}/regression_data/regression_data")

In [ ]:
filepaths = pd.Series(list(image_dir.glob(r'**/*.jpg')), name='Filepath').astype(str)
intensity = pd.Series(filepaths.apply(lambda x: os.path.split(os.path.split(x)[0])[1]), name='Intensity').astype(np.int32)

images = pd.concat([filepaths, intensity], axis=1).sample(frac=1.0, random_state=1).reset_index(drop=True)

In [ ]:
images

In [ ]:
image_df = images.sample(1844, random_state = 1).reset_index(drop = True)
train_df, test_df = train_test_split(image_df, train_size = 0.99, shuffle = True, random_state = 1)

In [ ]:
train_generator = ImageDataGenerator(rescale = 1./255, rotation_range = 5, validation_split = 0.2)
test_generator = ImageDataGenerator(rescale=1./255)

In [ ]:
train_df

In [ ]:
width = 310
height = 310

In [ ]:
train_images = train_generator.flow_from_dataframe(dataframe = train_df, 
                                                   x_col = 'Filepath', y_col = 'Intensity', 
                                                   target_size = (width, height),
                                                   class_mode = 'raw', batch_size = 4, shuffle = True, subset = 'training')

val_images = train_generator.flow_from_dataframe(dataframe = train_df,
                                                 x_col = 'Filepath', y_col = 'Intensity',
                                                 target_size = (width, height),
                                                 class_mode='raw', batch_size = 4, shuffle = True, subset = 'validation')

test_images = test_generator.flow_from_dataframe(dataframe = test_df,
                                                 x_col = 'Filepath', y_col = 'Intensity',
                                                 target_size = (width, height),
                                                 class_mode = 'raw', batch_size = 4, shuffle = False)

In [ ]:
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape = (width, height, 3), activation = 'relu'))
model.add(MaxPooling2D(pool_size = (2, 2)))
model.add(Dropout(0.2))

model.add(Conv2D(64, (3, 3), activation = 'relu'))
model.add(MaxPooling2D(pool_size = (2, 2)))
model.add(Dropout(0.2))

model.add(Conv2D(128, (3, 3), activation = 'relu'))
model.add(MaxPooling2D(pool_size = (2, 2)))
model.add(Dropout(0.2))

model.add(Conv2D(128, (3, 3), activation = 'relu'))
model.add(MaxPooling2D(pool_size = (2, 2)))
model.add(Dropout(0.2))

model.add(Flatten())

model.add(Dropout(0.3))

model.add(Dense(units = 512, activation = 'relu'))

model.add(Dropout(0.3))

model.add(Dense(1, activation = 'linear'))

In [ ]:
model.compile(optimizer = 'adam', loss = ['mse'])

history = model.fit(train_images, validation_data = val_images, epochs=100)

In [ ]:
model.save(f"{PROJECT_ROOT}/regression_model.keras")

# **Cascading**

In [ ]:
#regression_model = load_model(f'{PROJECT_ROOT}/regression_model.keras')
binary_model = load_model(f'{PROJECT_ROOT}/binary_model.keras')
MODEL_WEIGHTS_ROOT = PROJECT_ROOT + '/Model_Weights'
vgg_ensemble_model = load_model(f'{MODEL_WEIGHTS_ROOT}/multiclass_vgg_model.keras')
resnet_ensemble_model = load_model(f'{MODEL_WEIGHTS_ROOT}/multiclass_resnet_model.keras')
inception_ensemble_model = load_model(f'{MODEL_WEIGHTS_ROOT}/multiclass_inception_model.keras')

def ensemble_predict_class(img_batch):
    vgg_probs = vgg_ensemble_model.predict(img_batch, verbose = 0)
    resnet_probs = resnet_ensemble_model.predict(img_batch, verbose = 0)
    inception_probs = inception_ensemble_model.predict(img_batch, verbose = 0)
    avg_probs = (vgg_probs + resnet_probs + inception_probs) / 3.0
    pred_idx = int(np.argmax(avg_probs, axis = 1)[0])
    return pred_idx, avg_probs

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
from keras.preprocessing import image
print('Please upload an image')
uploaded = files.upload()

for fn in uploaded.keys():
  print('Image uploaded')
  # predicting images
  path = '/content/' + fn
  img = image.load_img(path, target_size=(310, 310))
  x = image.img_to_array(img)
  plt.imshow(x/255.)
  x = np.expand_dims(x, axis=0)
  images = np.vstack([x])
  classes = binary_model.predict(images, batch_size = 10)
  print(classes[0])

  if classes[0]<0.5:
    print("Uploaded image doesn't have a cyclone")

  else:
    print('Uploaded image has a cyclone')
    print('\nDetecting cyclone in the image.........\n')
    yolov3(path)
    if len(r) > 0 and r is not None:
      print('\n Cropped image\n')
      from IPython.display import display, Image
      display(Image(filename = f'{PROJECT_ROOT}/result.jpg'))

      regression_prediction = np.squeeze(regression_model.predict(x))

      x = image.img_to_array(image.load_img(f'{PROJECT_ROOT}/result.jpg', target_size=(310, 310)))
      x = np.expand_dims(x, axis=0)
      images = np.vstack([x])

      ensemble_class_idx, _ = ensemble_predict_class(x)
      output = {0: 'D', 1: 'DD', 2: 'CS', 3: 'SevereCS', 4: 'VSCS'}
      print('\nPredicted category of the cyclone in the image :-', output[ensemble_class_idx])

      #regression_prediction = np.squeeze(regression_model.predict(x))
      print('\nPredicted intensity of the cylone in the image :-', int(regression_prediction))
    
    else:
      ensemble_class_idx, _ = ensemble_predict_class(x)
      output = {0: 'D', 1: 'DD', 2: 'CS', 3: 'SevereCS', 4: 'VSCS'}
      print('\nPredicted category of the cyclone in the image :-', output[ensemble_class_idx])

**Another method**

In [ ]:
img_path = f'{PROJECT_ROOT}/binary_data_folders/test/yes/20001127.12-45.jpg'
img = image.load_img(img_path, target_size = (310, 310))
img_array = image.img_to_array(img)
img_batch = np.expand_dims(img_array, axis=0)
img_preprocessed = preprocess_input(img_batch)
binary_prediction = binary_model.predict(img_preprocessed)
if binary_prediction > 0:
    print('A cyclone is present')
    yolov3(img_path)
    ensemble_class_idx, _ = ensemble_predict_class(img_preprocessed)
    output = {0: 'D', 1: 'DD', 2: 'CS', 3: 'SevereCS', 4: 'VSCS'}
    print('Predicted intensity :-', output[ensemble_class_idx])
else:
    print('No')